In [1]:
# 标准库导入
import os
import json
import textwrap
import time
import zipfile
import io
import uuid
from pathlib import Path
from typing import Optional
from dataclasses import dataclass, field, asdict
from collections import defaultdict, Counter

print("标准库导入成功")

标准库导入成功


In [2]:
# 加载环境变量
try:
    from dotenv import load_dotenv
    load_dotenv()
    print("已加载 .env 配置文件")
except ImportError:
    print("未安装 python-dotenv，将使用系统环境变量")

已加载 .env 配置文件


In [3]:
# HTTP 请求库（用于调用 MinerU API）
import requests

# LangExtract 核心库
import langextract as lx
from langextract.providers.openai import OpenAILanguageModel
from langextract.prompt_validation import PromptValidationLevel

print("LangExtract 导入成功")
print(f"  版本信息: {lx.__version__ if hasattr(lx, '__version__') else '未知'}")


LangExtract 导入成功
  版本信息: 未知


In [4]:
# ChromaDB 向量数据库
import chromadb
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

print("ChromaDB 和 LangChain 导入成功")

ChromaDB 和 LangChain 导入成功


In [5]:
# MinerU API 配置
MINERU_API_KEY = os.getenv("MINERU_API_KEY", "")
MINERU_BASE_URL = "https://mineru.net"

print("MinerU 配置:")
if MINERU_API_KEY:
    print(f"  API Key: {MINERU_API_KEY[:10]}... (已配置)")
    print(f"  API URL: {MINERU_BASE_URL}")
else:
    print(" API Key 未配置")

MinerU 配置:
  API Key: sk-KJVtOkw... (已配置)
  API URL: https://mineru.net


In [6]:
# DeepSeek LLM 配置
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY", "")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com")
DEFAULT_MODEL = os.getenv("DEFAULT_MODEL", "deepseek-v4-flash")

print("\nDeepSeek 配置:")
if DEEPSEEK_API_KEY:
    print(f"  API Key: {DEEPSEEK_API_KEY[:10]}... (已配置)")
    print(f"  API URL: {DEEPSEEK_BASE_URL}")
    print(f"  模型: {DEFAULT_MODEL}")
else:
    print(" API Key 未配置")


DeepSeek 配置:
  API Key: sk-66a9cec... (已配置)
  API URL: https://api.deepseek.com
  模型: deepseek-v4-flash


In [7]:
# 阿里云百炼 Embeddings 配置
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY", "")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL", "https://dashscope.aliyuncs.com/compatible-mode/v1")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-v1")

print("\n阿里云百炼 Embeddings 配置:")
if DASHSCOPE_API_KEY:
    print(f"  API Key: {DASHSCOPE_API_KEY[:10]}... (已配置)")
    print(f"  API URL: {DASHSCOPE_BASE_URL}")
    print(f"  Embedding 模型: {EMBEDDING_MODEL}")
else:
    print(" API Key 未配置")


阿里云百炼 Embeddings 配置:
  API Key: sk-8adbc1f... (已配置)
  API URL: https://dashscope.aliyuncs.com/compatible-mode/v1
  Embedding 模型: text-embedding-v1


In [8]:
# ChromaDB 配置
CHROMA_PERSIST_DIR = "./chroma_db_chapter5"
COLLECTION_NAME = "pdf_knowledge_graph"

# 输出目录配置
OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("\n存储配置:")
print(f"  ChromaDB 存储路径: {CHROMA_PERSIST_DIR}")
print(f"  集合名称: {COLLECTION_NAME}")
print(f"  输出目录: {OUTPUT_DIR}")


存储配置:
  ChromaDB 存储路径: ./chroma_db_chapter5
  集合名称: pdf_knowledge_graph
  输出目录: outputs


In [9]:
# 指定要解析的 PDF 文件
PDF_PATH = "../test.pdf"  # 修改为你的 PDF 文件路径

# 检查文件是否存在
pdf_file = Path(PDF_PATH)
if pdf_file.exists():
    file_size = pdf_file.stat().st_size
    print(f"PDF 文件存在: {PDF_PATH}")
    print(f"  文件大小: ({file_size/1024:.2f} KB)")
else:
    print(f"PDF 文件不存在: {PDF_PATH}")
    print("  请修改 PDF_PATH 变量为有效的文件路径")

PDF 文件存在: ../test.pdf
  文件大小: (87.89 KB)


In [10]:
# 记录开始时间
parse_start_time = time.time()

# 准备 HTTP 请求头
headers = {
    "Authorization": f"Bearer {MINERU_API_KEY}",
    "Content-Type": "application/json",
}

# 步骤1：请求预签名上传 URL
print("步骤1：请求上传 URL...")
response = requests.post(
    f"{MINERU_BASE_URL}/api/v4/file-urls/batch",
    headers=headers,
    json={
        "files": [{"name": pdf_file.name, "data_id": pdf_file.name}],
        "model_version": "vlm",  # 使用视觉语言模型版本
    }
)
response.raise_for_status()  # 如果请求失败，抛出异常

# 解析响应
data = response.json()["data"]
batch_id = data["batch_id"]
upload_url = data["file_urls"][0]

print(f"  批次 ID: {batch_id}")
print(f"  上传 URL: {upload_url[:50]}...")

步骤1：请求上传 URL...
  批次 ID: 77b81945-a43a-4f77-ad4a-32bbd5fef5fd
  上传 URL: https://mineru.oss-cn-shanghai.aliyuncs.com/api-up...


In [11]:
# 步骤2：上传 PDF 文件
print("\n步骤2：上传 PDF...")

with open(PDF_PATH, "rb") as f:
    file_content = f.read()
    upload_response = requests.put(upload_url, data=file_content)
    upload_response.raise_for_status()

print(f"   上传速度: {len(file_content) / (time.time() - parse_start_time) / 1024:.2f} KB/s")


步骤2：上传 PDF...
   上传速度: 15.76 KB/s


In [12]:
# 步骤3：轮询等待解析
print("\n步骤3：等待解析...")

max_wait = 600  # 最长等待时间（秒）
poll_interval = 3  # 轮询间隔（秒）
elapsed = 0
full_zip_url = None

while elapsed < max_wait:
    # 查询解析状态
    status_response = requests.get(
        f"{MINERU_BASE_URL}/api/v4/extract-results/batch/{batch_id}",
        headers={"Authorization": f"Bearer {MINERU_API_KEY}"}
    )
    result = status_response.json()["data"]["extract_result"][0]
    state = result["state"]
    
    if state == "done":
        # 解析完成
        full_zip_url = result["full_zip_url"]
        print(f"   ✓ 解析完成！(耗时 {elapsed}秒)")
        break
    elif state == "failed":
        # 解析失败
        error_msg = result.get('err_msg', '未知错误')
        print(f"   解析失败: {error_msg}")
        break
    elif state == "running":
        # 正在解析
        progress = result.get("extract_progress", {})
        extracted = progress.get("extracted_pages", 0)
        total = progress.get("total_pages", "?")
        print(f"   {extracted}/{total} 页 (已等待 {elapsed}秒)", end="\r")
    
    time.sleep(poll_interval)
    elapsed += poll_interval

if elapsed >= max_wait:
    print(f"\n   解析超时 (超过 {max_wait}秒)")


步骤3：等待解析...
   ✓ 解析完成！(耗时 0秒)


In [13]:
# 步骤4：下载并提取 Markdown
if full_zip_url:
    print("\n步骤4：下载解析结果...")
    
    # 下载 ZIP 包
    zip_response = requests.get(full_zip_url)
    
    # 解压并提取 Markdown 文件
    with zipfile.ZipFile(io.BytesIO(zip_response.content)) as zf:
        # 查找 .md 文件
        md_files = [f for f in zf.namelist() if f.endswith('.md')]
        
        if md_files:
            # 优先选择文件名包含 'full' 的文件，否则选第一个
            target_file = next((f for f in md_files if 'full' in f.lower()), md_files[0])
            
            # 读取 Markdown 内容
            with zf.open(target_file) as f:
                markdown_text = f.read().decode('utf-8')
            
            print(f"   成功提取 Markdown: {target_file}")
            print(f"   Markdown 长度: {len(markdown_text):,} 字符")
        else:
            print("   ZIP 包中未找到 .md 文件")
            markdown_text = None
    
    # 统计解析总耗时
    parse_time = time.time() - parse_start_time
    print(f"\nPDF 解析完成 (总耗时 {parse_time:.2f}秒)")
else:
    print("\n未能获取解析结果")
    markdown_text = None


步骤4：下载解析结果...
   成功提取 Markdown: full.md
   Markdown 长度: 683 字符

PDF 解析完成 (总耗时 29.47秒)


In [14]:
# 保存 Markdown 到文件
if markdown_text:
    md_file = OUTPUT_DIR / "parsed_document.md"
    with open(md_file, "w", encoding="utf-8") as f:
        f.write(markdown_text)
    
    print(f"Markdown 已保存: {md_file}")
    print(f"\nMarkdown 预览（前500字符）:")
    print("-" * 80)
    print(markdown_text[:500])
    print("-" * 80)
else:
    print("没有 Markdown 内容可保存")

Markdown 已保存: outputs\parsed_document.md

Markdown 预览（前500字符）:
--------------------------------------------------------------------------------
## 【案件基本信息】

案号：（2024）京0105民初12345号

案由：民间借贷纠纷

原告：张三

被告：李四

## 【案件事实】

2023年1月15日，被告李四因资金周转需要向原告张三借款人民币50万元，

双方签订《借款合同》，约定借款期限为一年，年利率为12%，

到期日为2024年1月15日。借款到期后，被告仅偿还本金10万元，

剩余40万元本金及利息至今未还。原告多次催要未果，遂诉至法院。

## 【争议焦点】

## 1. 借款合同的效力问题

利息的计算标准

## 3. 被告是否应当承担违约责任

【法院认为】

本院认为，原、被告之间的借款合同系双方真实意思表示，不违反法律、

行政法规的强制性规定，合法有效。根据《中华人民共和国民法典》

第六百七十五条规定，借款人应当按照约定的期限返还借款。

被告未按期足额还款，构成违约，应承担相应的违约责任。

关于利息，根据《最高人民法院关于审理民间借贷案件适用法律若干问题的规定》

第二十五条，年利率12%未超过合同成立时一年期贷款市场报价利率四倍，

应予支持。

## 【判决结果】

一、被告李四于本判
--------------------------------------------------------------------------------


In [15]:
# 定义知识提取的 Prompt
extraction_prompt = textwrap.dedent("""\
    从文档中提取以下结构化知识:
    
    - 实体: 人物、机构、地点、时间、概念、技术术语
    - 数据指标: 数值、百分比、统计数据
    - 关系描述: 实体之间的关系（合作、隶属、引用等）
    - 事件: 重要事件和行为
    
    要求:
    1. extraction_text 必须是原文的精确子串
    2. 为每个提取添加丰富的属性信息
    3. 关系类型必须在 attributes 中标注涉及的主体
    4. 保持原文出现顺序
    """)

print("提取 Prompt 定义完成")
print("\nPrompt 内容:")
print(extraction_prompt)

提取 Prompt 定义完成

Prompt 内容:
从文档中提取以下结构化知识:

- 实体: 人物、机构、地点、时间、概念、技术术语
- 数据指标: 数值、百分比、统计数据
- 关系描述: 实体之间的关系（合作、隶属、引用等）
- 事件: 重要事件和行为

要求:
1. extraction_text 必须是原文的精确子串
2. 为每个提取添加丰富的属性信息
3. 关系类型必须在 attributes 中标注涉及的主体
4. 保持原文出现顺序



In [16]:
# 定义 Few-shot 示例
example_text = textwrap.dedent("""\
    # 人工智能发展报告
    
    ## 作者
    张三（清华大学）、李四（北京大学）
    
    ## 摘要
    本报告分析了2024年人工智能的发展趋势。研究显示，大模型参数量增长了300%，
    推理成本下降了50%。清华大学与北京大学联合开展了此项研究。
    """).strip()

# 期望的提取结果
example_extractions = [
    lx.data.Extraction(
        extraction_class="实体",
        extraction_text="张三",
        attributes={"类型": "人物", "机构": "清华大学"}
    ),
    lx.data.Extraction(
        extraction_class="实体",
        extraction_text="清华大学",
        attributes={"类型": "机构", "类别": "高校"}
    ),
    lx.data.Extraction(
        extraction_class="数据指标",
        extraction_text="增长了300%",
        attributes={"指标": "大模型参数量", "类型": "增长率"}
    ),
    lx.data.Extraction(
        extraction_class="关系描述",
        extraction_text="清华大学与北京大学联合开展了此项研究",
        attributes={
            "类型": "合作关系",
            "主体1": "清华大学",
            "主体2": "北京大学",
            "关系": "联合研究"
        }
    ),
]

# 组装为 ExampleData
examples = [
    lx.data.ExampleData(
        text=example_text,
        extractions=example_extractions
    )
]

print("Few-shot 示例定义完成")
print(f"\n示例文本长度: {len(example_text)} 字符")
print(f"示例提取数量: {len(example_extractions)} 个")

Few-shot 示例定义完成

示例文本长度: 114 字符
示例提取数量: 4 个


In [17]:
# 创建 LangExtract 模型
langextract_model = OpenAILanguageModel(
    model_id=DEFAULT_MODEL,
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)

print("LangExtract 模型创建成功")
print(f"  模型: {DEFAULT_MODEL}")
print(f"  API: {DEEPSEEK_BASE_URL}")

# 准备提取参数
doc_id = "doc_001"
doc_title = PDF_PATH

# 如果 Markdown 文本过长，截断到 8000 字符（避免超过 LLM 上下文限制）
max_length = 8000
if markdown_text and len(markdown_text) > max_length:
    print(f"\n Markdown 文本较长 ({len(markdown_text):,} 字符)，截断至 {max_length:,} 字符")
    extraction_text = markdown_text[:max_length]
else:
    extraction_text = markdown_text

print(f"\n准备提取:")
print(f"  文档 ID: {doc_id}")
print(f"  文档标题: {doc_title}")
print(f"  文本长度: {len(extraction_text):,} 字符" if extraction_text else "  文本长度: 0 字符")

LangExtract 模型创建成功
  模型: deepseek-v4-flash
  API: https://api.deepseek.com

准备提取:
  文档 ID: doc_001
  文档标题: ../test.pdf
  文本长度: 683 字符


In [18]:
# 执行 LangExtract 提取
print("\n正在执行 LangExtract 提取...\n")
extract_start_time = time.time()

if extraction_text:
    result = lx.extract(
        text_or_documents=extraction_text,
        prompt_description=extraction_prompt,
        examples=examples,
        model=langextract_model,
        fence_output=True,  # 要求 LLM 输出用代码块包裹，避免格式错误
        use_schema_constraints=False,  # 不使用严格的 schema 约束，提高灵活性
        prompt_validation_level=PromptValidationLevel.OFF,  # 关闭提示词验证
        show_progress=True,  # 显示提取进度
    )
    
    extract_time = time.time() - extract_start_time
    print(f"\n提取完成 (耗时 {extract_time:.2f}秒)")
else:
    print("没有可提取的文本")
    result = None


正在执行 LangExtract 提取...



LangExtract: model=deepseek-v4-flash, current=683 chars, processed=0 chars:  [00:49]


提取完成 (耗时 49.27秒)


In [19]:
# 定义知识提取结果的数据结构
@dataclass
class KnowledgeExtraction:
    """知识提取结果（带溯源）"""
    doc_id: str                              # 文档 ID
    doc_title: str                           # 文档标题
    extraction_class: str                    # 提取类型（实体、关系、事件等）
    extraction_text: str                     # 提取的原文文本
    char_interval: Optional[dict] = None     # 原文字符区间（溯源的核心）
    attributes: dict = field(default_factory=dict)  # 属性信息
    
    def to_dict(self) -> dict:
        """转换为字典格式"""
        return asdict(self)
    
    def to_searchable_text(self) -> str:
        """生成用于向量化的可搜索文本"""
        parts = [
            f"类型: {self.extraction_class}",
            f"内容: {self.extraction_text}",
            f"来源: {self.doc_title}"
        ]
        if self.attributes:
            for k, v in self.attributes.items():
                parts.append(f"{k}: {v}")
        return " | ".join(parts)

print("KnowledgeExtraction 数据结构定义完成")
print("\n字段说明:")
print("  - doc_id: 文档唯一标识")
print("  - doc_title: 文档名称")
print("  - extraction_class: 实体类型（如'实体'、'关系描述'、'数据指标'）")
print("  - extraction_text: 从原文提取的精确文本")
print("  - char_interval: {'start_pos': 起始位置, 'end_pos': 结束位置} ← 溯源关键")
print("  - attributes: 实体的属性信息（如类型、角色等）")


KnowledgeExtraction 数据结构定义完成

字段说明:
  - doc_id: 文档唯一标识
  - doc_title: 文档名称
  - extraction_class: 实体类型（如'实体'、'关系描述'、'数据指标'）
  - extraction_text: 从原文提取的精确文本
  - char_interval: {'start_pos': 起始位置, 'end_pos': 结束位置} ← 溯源关键
  - attributes: 实体的属性信息（如类型、角色等）


In [20]:
# 转换为 KnowledgeExtraction 对象列表
extractions = []

if result:
    for ext in result.extractions:
        # 提取 char_interval（溯源的核心）
        if ext.char_interval:
            char_interval = {
                "start_pos": ext.char_interval.start_pos,
                "end_pos": ext.char_interval.end_pos
            }
        else:
            char_interval = None
        
        # 创建 KnowledgeExtraction 对象
        ke = KnowledgeExtraction(
            doc_id=doc_id,
            doc_title=doc_title,
            extraction_class=ext.extraction_class,
            extraction_text=ext.extraction_text,
            char_interval=char_interval,
            attributes=ext.attributes or {}
        )
        extractions.append(ke)

    print(f"已转换为 KnowledgeExtraction 对象")
    print(f"  提取实体总数: {len(extractions)}")
else:
    print("没有提取结果")

已转换为 KnowledgeExtraction 对象
  提取实体总数: 21


In [21]:
# 统计实体类型分布
if extractions:
    type_counts = Counter(e.extraction_class for e in extractions)
    
    print("\n提取结果统计:")
    print("\n实体类型分布:")
    for entity_type, count in type_counts.most_common():
        percentage = count / len(extractions) * 100
        print(f"   - {entity_type}: {count} 个 ({percentage:.1f}%)")
    
    # 统计溯源情况
    aligned_count = sum(1 for e in extractions if e.char_interval)
    alignment_rate = aligned_count / len(extractions) * 100 if extractions else 0
    
    print(f"\n溯源统计:")
    print(f"   - 有原文位置: {aligned_count} 个")
    print(f"   - 无原文位置: {len(extractions) - aligned_count} 个")
    print(f"   - 对齐率: {alignment_rate:.1f}%")
    
    if alignment_rate < 80:
        print("   对齐率较低，可能影响溯源质量")
    else:
        print("   对齐率良好，溯源质量较高")
    
    # 统计属性情况
    with_attrs = sum(1 for e in extractions if e.attributes)
    attrs_rate = with_attrs / len(extractions) * 100 if extractions else 0
    
    print(f"\n属性统计:")
    print(f"   - 有属性: {with_attrs} 个")
    print(f"   - 无属性: {len(extractions) - with_attrs} 个")
    print(f"   - 属性率: {attrs_rate:.1f}%")
    
    if attrs_rate < 60:
        print("   属性较少，建议优化 Few-shot 示例")
    else:
        print("   属性丰富，有利于知识图谱构建")
else:
    print("没有提取结果可统计")



提取结果统计:

实体类型分布:
   - 实体: 9 个 (42.9%)
   - 事件: 5 个 (23.8%)
   - 数据指标: 5 个 (23.8%)
   - 关系描述: 2 个 (9.5%)

溯源统计:
   - 有原文位置: 18 个
   - 无原文位置: 3 个
   - 对齐率: 85.7%
   对齐率良好，溯源质量较高

属性统计:
   - 有属性: 21 个
   - 无属性: 0 个
   - 属性率: 100.0%
   属性丰富，有利于知识图谱构建


In [22]:
# 展示前5个提取示例（含溯源验证）
if extractions and extraction_text:
    print("\n提取示例（前5个，含溯源验证）:")
    print("=" * 80)
    
    for i, ext in enumerate(extractions[:5], 1):
        print(f"\n[{i}] {ext.extraction_class}: {ext.extraction_text[:50]}{'...' if len(ext.extraction_text) > 50 else ''}")
        
        if ext.attributes:
            print(f"    属性: {ext.attributes}")
        
        if ext.char_interval:
            interval = ext.char_interval
            print(f"    原文位置: 字符 {interval['start_pos']} - {interval['end_pos']}")
            
            # 验证：从 Markdown 中提取对应位置的文本
            original_text = extraction_text[interval['start_pos']:interval['end_pos']]
            if original_text == ext.extraction_text:
                print(f"    溯源验证通过")
            else:
                print(f"    溯源不匹配: 原文为 '{original_text[:30]}...'")
        else:
            print(f"    无原文位置信息")
    
    print("\n" + "=" * 80)
    
    # 保存提取结果
    extractions_file = OUTPUT_DIR / "chapter5_extractions.json"
    with open(extractions_file, "w", encoding="utf-8") as f:
        json.dump([e.to_dict() for e in extractions], f, ensure_ascii=False, indent=2)
    
    print(f"\n提取结果已保存: {extractions_file}")
    print(f"   文件大小: {extractions_file.stat().st_size / 1024:.2f} KB")
else:
    print("没有提取结果可展示")



提取示例（前5个，含溯源验证）:

[1] 实体: （2024）京0105民初12345号
    属性: {'类型': '案号', '法院代码': '京0105', '年度': '2024', '案件类别': '民初', '序号': '12345'}
    原文位置: 字符 16 - 35
    溯源验证通过

[2] 实体: 民间借贷纠纷
    属性: {'类型': '案由', '性质': '民事纠纷'}
    原文位置: 字符 40 - 46
    溯源验证通过

[3] 实体: 张三
    属性: {'类型': '人物', '角色': '原告'}
    原文位置: 字符 51 - 53
    溯源验证通过

[4] 实体: 李四
    属性: {'类型': '人物', '角色': '被告'}
    原文位置: 字符 58 - 60
    溯源验证通过

[5] 实体: 2023年1月15日
    属性: {'类型': '时间', '对应事件': '借款日期'}
    原文位置: 字符 73 - 83
    溯源验证通过


提取结果已保存: outputs\chapter5_extractions.json
   文件大小: 7.88 KB


In [23]:
# 初始化 Embeddings
embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    check_embedding_ctx_length=False,  # 禁用 token 长度检查
    chunk_size=10,  # 阿里云百炼限制批量大小不超过 10
)
print(f"Embeddings: {EMBEDDING_MODEL}")

# 初始化 ChromaDB
persist_dir = Path(CHROMA_PERSIST_DIR)
persist_dir.mkdir(parents=True, exist_ok=True)
client = chromadb.PersistentClient(path=str(persist_dir))
print(f"ChromaDB: {persist_dir}")

# 重建集合（删除旧数据）
try:
    client.delete_collection(name=COLLECTION_NAME)
    print(f"  已删除旧集合")
except:
    pass

# 创建 LangChain 的 Chroma 向量存储
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    client=client,
)
print(f"  已创建集合: {COLLECTION_NAME}\n")

Embeddings: text-embedding-v1
ChromaDB: chroma_db_chapter5
  已创建集合: pdf_knowledge_graph



In [24]:
# 准备向量化数据
if extractions:
    print(f"存储到 ChromaDB...")
    
    texts = [ext.to_searchable_text() for ext in extractions]
    metadatas = []
    ids = []
    
    for ext in extractions:
        ids.append(str(uuid.uuid4()))
        # 关键：在 metadata 中保存溯源信息
        metadatas.append({
            "doc_id": ext.doc_id,
            "doc_title": ext.doc_title,
            "extraction_class": ext.extraction_class,
            "extraction_text": ext.extraction_text,
            "char_interval": json.dumps(ext.char_interval) if ext.char_interval else "",  # 溯源关键
            "attributes": json.dumps(ext.attributes, ensure_ascii=False),
        })
    
    # 批量插入向量
    vectorstore.add_texts(texts=texts, metadatas=metadatas, ids=ids)
    print(f"✓ 已索引 {len(texts)} 条记录\n")
    
    # 查询集合信息
    collection = client.get_collection(COLLECTION_NAME)
    count = collection.count()
    print(f"集合统计:")
    print(f"   - 向量数量: {count}")
    print(f"   - 集合名称: {COLLECTION_NAME}")
else:
    print("没有提取结果可存储")

存储到 ChromaDB...
✓ 已索引 21 条记录

集合统计:
   - 向量数量: 21
   - 集合名称: pdf_knowledge_graph


In [25]:
# 构建知识图谱
knowledge_graph = {
    "entities": {},
    "relations": []
}

if extractions:
    for ext in extractions:
        if ext.extraction_class == "关系描述":
            # 提取关系
            attrs = ext.attributes
            knowledge_graph["relations"].append({
                "text": ext.extraction_text,
                "type": attrs.get("类型", "未知"),
                "subject": attrs.get("主体1"),
                "object": attrs.get("主体2"),
                "relation": attrs.get("关系"),
                "source": ext.doc_title
            })
        elif ext.extraction_class in ["实体", "数据指标"]:
            # 提取实体（保留溯源信息）
            entity_name = ext.extraction_text
            if entity_name not in knowledge_graph["entities"]:
                knowledge_graph["entities"][entity_name] = {
                    "type": ext.extraction_class,
                    "attributes": ext.attributes,
                    "mentions": []  # 存储所有提及位置（溯源）
                }
            # 添加提及位置
            knowledge_graph["entities"][entity_name]["mentions"].append({
                "source": ext.doc_title,
                "position": ext.char_interval  # 溯源关键
            })
    
    print(f"知识图谱统计:")
    print(f"   - 实体数: {len(knowledge_graph['entities'])}")
    print(f"   - 关系数: {len(knowledge_graph['relations'])}")
    
    # 保存知识图谱
    kg_file = OUTPUT_DIR / "knowledge_graph.json"
    with open(kg_file, "w", encoding="utf-8") as f:
        json.dump(knowledge_graph, f, ensure_ascii=False, indent=2)
    print(f"\n知识图谱已保存: {kg_file}")
    
    # 展示部分知识图谱（含溯源信息）
    print(f"\n实体示例（前5个，含溯源信息）:")
    for i, (entity, data) in enumerate(list(knowledge_graph['entities'].items())[:5], 1):
        print(f"   [{i}] {entity}")
        print(f"       类型: {data['type']}")
        if data['attributes']:
            print(f"       属性: {data['attributes']}")
        if data['mentions']:
            print(f"       提及次数: {len(data['mentions'])}")
            for j, mention in enumerate(data['mentions'][:2], 1):
                if mention.get('position'):
                    pos = mention['position']
                    print(f"         提及{j}: 字符 {pos['start_pos']}-{pos['end_pos']} (来源: {mention['source']})")
else:
    print("没有提取结果可构建知识图谱")

知识图谱统计:
   - 实体数: 14
   - 关系数: 2

知识图谱已保存: outputs\knowledge_graph.json

实体示例（前5个，含溯源信息）:
   [1] （2024）京0105民初12345号
       类型: 实体
       属性: {'类型': '案号', '法院代码': '京0105', '年度': '2024', '案件类别': '民初', '序号': '12345'}
       提及次数: 1
         提及1: 字符 16-35 (来源: ../test.pdf)
   [2] 民间借贷纠纷
       类型: 实体
       属性: {'类型': '案由', '性质': '民事纠纷'}
       提及次数: 1
         提及1: 字符 40-46 (来源: ../test.pdf)
   [3] 张三
       类型: 实体
       属性: {'类型': '人物', '角色': '原告'}
       提及次数: 1
         提及1: 字符 51-53 (来源: ../test.pdf)
   [4] 李四
       类型: 实体
       属性: {'类型': '人物', '角色': '被告'}
       提及次数: 1
         提及1: 字符 58-60 (来源: ../test.pdf)
   [5] 2023年1月15日
       类型: 实体
       属性: {'类型': '时间', '对应事件': '借款日期'}
       提及次数: 1
         提及1: 字符 73-83 (来源: ../test.pdf)
